In [1]:
import dcor
import pickle
import numpy as np
import os
from weightedcorrs import weightedcorrs




In [6]:
import torch

def distance_corr(var_1,var_2,normedweight,power=1):
    """var_1: First variable to decorrelate (eg mass)
    var_2: Second variable to decorrelate (eg classifier output)
    normedweight: Per-example weight. Sum of weights should add up to N (where N is the number of examples)
    power: Exponent used in calculating the distance correlation
    
    va1_1, var_2 and normedweight should all be 1D torch tensors with the same number of entries
    
    Usage: Add to your loss function. total_loss = BCE_loss + lambda * distance_corr
    """
    
    
    xx = var_1.view(-1, 1).repeat(1, len(var_1)).view(len(var_1),len(var_1))
    yy = var_1.repeat(len(var_1),1).view(len(var_1),len(var_1))
    amat = (xx-yy).abs()

    xx = var_2.view(-1, 1).repeat(1, len(var_2)).view(len(var_2),len(var_2))
    yy = var_2.repeat(len(var_2),1).view(len(var_2),len(var_2))
    bmat = (xx-yy).abs()

    amatavg = torch.mean(amat*normedweight,dim=1)
    Amat=amat-amatavg.repeat(len(var_1),1).view(len(var_1),len(var_1))\
        -amatavg.view(-1, 1).repeat(1, len(var_1)).view(len(var_1),len(var_1))\
        +torch.mean(amatavg*normedweight)

    bmatavg = torch.mean(bmat*normedweight,dim=1)
    Bmat=bmat-bmatavg.repeat(len(var_2),1).view(len(var_2),len(var_2))\
        -bmatavg.view(-1, 1).repeat(1, len(var_2)).view(len(var_2),len(var_2))\
        +torch.mean(bmatavg*normedweight)

    ABavg = torch.mean(Amat*Bmat*normedweight,dim=1)
    AAavg = torch.mean(Amat*Amat*normedweight,dim=1)
    BBavg = torch.mean(Bmat*Bmat*normedweight,dim=1)

    if(power==1):
        dCorr=(torch.mean(ABavg*normedweight))/torch.sqrt((torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight)))
    elif(power==2):
        dCorr=(torch.mean(ABavg*normedweight))**2/(torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight))
    else:
        dCorr=((torch.mean(ABavg*normedweight))/torch.sqrt((torch.mean(AAavg*normedweight)*torch.mean(BBavg*normedweight))))**power
    
    return dCorr

# Compute DisCo between score and mass

In [2]:
name = ""
cwd = os.getcwd()


In [3]:
with open(cwd + "/output/LundNet5+_SVJ-QCD-TTJet-EW_vs12/_eval_ROC_AUC.pkl", 'rb') as file:
    evaluation_dict = pickle.load(file)

In [12]:
score = []
mt = []
weights = []

key = "SVJ_1500_0p3_LN5p"

for n,label in enumerate(evaluation_dict[key]["dataset_labels"]):
    if "Z" in label:
        score.append(evaluation_dict[key]["scores"][n])
        mt.append(evaluation_dict[key]["MT01FatJet"][n])
        weights.append(evaluation_dict[key]["weight"][n])
    if label == "QCD":
        print("wrong key in dict")


In [13]:
correlation = weightedcorrs(np.transpose([np.array(mt), np.array(score)]), np.array(weights))
print(correlation["R"][0,1])

-0.04388077


In [11]:
weights = weights/sum(weights) * len(weights)

In [12]:
len(weights)

279627

In [7]:
import torch

In [38]:
import random


# Combining the arrays into a list of tuples
combined = list(zip(mt, score, weights))

# Shuffling the combined list
random.shuffle(combined)

# Unzipping the shuffled list back into separate arrays
mt_shuffled, score_shuffled, weights_shuffled = zip(*combined)

# Converting the tuples back into lists
mt_shuffled = list(mt_shuffled)
score_shuffled = list(score_shuffled)
weights_shuffled = list(weights_shuffled)




In [6]:
torch.set_num_threads(10)
disco = distance_corr(torch.tensor(mt_shuffled[:50000]),torch.tensor(score_shuffled[:50000]),torch.tensor(weights_shuffled[:50000]))
# disco = dcor.distance_correlation(np.array(score),np.array(mt))

NameError: name 'torch' is not defined

In [42]:
disco

tensor(0.0032, dtype=torch.float64)

In [24]:
disco

tensor(0.0022, dtype=torch.float64)